# BAN (Body Area Network) System

## Database & Real-Time Communication Design Document

---

## 1. Project Context

The BAN (Body Area Network) system is a healthcare monitoring platform designed to collect, process, and visualize **time-sensitive physiological data** from patients using **ESP32-based wearable sensors**. The system supports:

* Real-time vitals monitoring (HR, SpO₂, Temperature, BP)
* Continuous historical data storage
* Alert generation
* Predictive risk analysis
* Live dashboards for medical staff

Because this is a **medical-grade, real-time system**, technology choices must prioritize **reliability, data integrity, scalability, and correctness**.

---

## 2. Database Selection

### 2.1 Chosen Database: PostgreSQL (SQL)

**PostgreSQL** is selected as the primary database for the BAN system.

PostgreSQL is a **relational SQL database** known for its strong consistency, reliability, and ability to handle structured and time-series data efficiently.

---

### 2.2 Why PostgreSQL is Used

#### a) Structured Medical Data

Healthcare data is highly structured and relational:

* Patients
* Staff
* Devices
* Vitals
* Alerts
* Risk assessments

PostgreSQL enforces **schemas, constraints, and relationships**, ensuring:

* No orphan records
* No invalid values
* Strong data integrity

This is critical for medical systems where incorrect or inconsistent data can lead to wrong decisions.

---

#### b) Time-Sensitive & Time-Series Support

BAN data is time-based:

* Heart rate every few seconds
* SpO₂ trends over hours
* BP readings at intervals
* Alerts at exact timestamps

PostgreSQL handles this using:

* `TIMESTAMP WITH TIME ZONE`
* Indexed time-based queries
* Aggregations (AVG, MIN, MAX, trends)

Typical queries include:

* Last 24 hours of vitals
* Hourly averages
* Trend detection

These are native SQL operations in PostgreSQL.

---

#### c) ACID Compliance (Critical for Healthcare)

PostgreSQL is fully **ACID compliant**:

* **Atomicity** – data writes are all-or-nothing
* **Consistency** – rules and constraints are enforced
* **Isolation** – concurrent sensor writes do not corrupt data
* **Durability** – once saved, data persists safely

This makes PostgreSQL suitable for **medical-grade systems**.

---

#### d) Predictive Analytics & ML Friendly

The BAN system includes predictive risk analysis.

PostgreSQL integrates seamlessly with:

* Python
* FastAPI
* Pandas
* Machine Learning pipelines

Data can be:

* Extracted for preprocessing
* Used for model training
* Written back as risk scores

---

### 2.3 Comparison with Other Databases

#### MongoDB (NoSQL)

* Weak schema enforcement
* Poor relational integrity
* Harder to manage time-series analytics
* Not ideal for medical data

❌ Not recommended

---

#### Firebase / Realtime DB

* Limited querying
* Poor analytical capabilities
* Vendor lock-in
* Not designed for heavy sensor analytics

❌ Not suitable

---

#### MySQL

* Good relational database
* Fewer advanced features than PostgreSQL
* Weaker support for analytics and extensibility

⚠️ Acceptable, but PostgreSQL is superior

---

### 2.4 Final Database Decision

**PostgreSQL is chosen because it provides the best balance of:**

* Data integrity
* Time-series support
* Scalability
* Analytics readiness
* Industry acceptance in healthcare

---

## 3. Real-Time Communication Design

The BAN system requires **real-time data flow** at two different levels:

1. Sensor → Backend (IoT level)
2. Backend → Dashboard (Application/UI level)

Using a single protocol for both is inefficient and unreliable.

Therefore, a **hybrid communication architecture** is used.

---

## 4. MQTT (ESP32 → Backend)

### 4.1 What is MQTT?

MQTT (Message Queuing Telemetry Transport) is a **lightweight publish–subscribe protocol** designed specifically for:

* IoT devices
* Low-power hardware
* Unstable networks

---

### 4.2 Why MQTT is Used for ESP32

ESP32 devices:

* Have limited RAM and CPU
* Operate on intermittent networks
* Continuously send sensor data

MQTT provides:

* Minimal overhead
* Automatic reconnection
* Message buffering
* Reliable delivery using QoS levels

This makes MQTT ideal for **medical sensor communication**.

---

### 4.3 Handling Critical Data (e.g. Blood Pressure)

Blood pressure readings are:

* Not continuous
* Clinically critical
* Must not be lost

MQTT supports **Quality of Service (QoS)**:

* QoS 0 – fire and forget
* QoS 1 – delivered at least once
* QoS 2 – delivered exactly once

For BP:

* QoS 1 or QoS 2 is used
* Backend confirms receipt

This ensures reliability.

---

### 4.4 MQTT Flow

* ESP32 publishes vitals to MQTT topics
* MQTT broker (e.g. Mosquitto) receives data
* Backend subscribes to topics
* Data is validated and processed

---

## 5. WebSockets (Backend → Dashboard)

### 5.1 What are WebSockets?

WebSockets provide a **persistent, bi-directional connection** between:

* Backend server
* Web browser

They are ideal for real-time UI updates.

---

### 5.2 Why WebSockets are Used for Dashboards

Dashboards require:

* Live vitals updates
* Instant alerts
* Smooth charts and graphs

WebSockets allow:

* Server push (no polling)
* Low-latency updates
* Efficient UI synchronization

Browsers handle WebSockets efficiently, unlike microcontrollers.

---

### 5.3 Why WebSockets are NOT Used for ESP32

* Heavy protocol
* Stateful connections
* Memory-intensive
* Poor reconnection handling on IoT devices

❌ Not suitable for sensors

---

## 6. Combined System Architecture

The final communication architecture:

ESP32 Sensors
→ MQTT
→ MQTT Broker
→ FastAPI Backend
→ PostgreSQL (storage)
→ Risk Analysis Engine
→ WebSockets
→ Live Dashboard (Next.js)

---

## 7. Key Design Advantages

* Reliable sensor communication
* Guaranteed delivery of critical vitals
* Real-time UI updates
* Strong data integrity
* Scalable and industry-aligned architecture

---

## 8. Final Summary

* **PostgreSQL (SQL)** is used for reliable, time-sensitive medical data storage
* **MQTT** is used for ESP32 sensor communication
* **WebSockets** are used for real-time dashboards


